In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D # For custom legends
import numpy as np
import os
import glob

# For map projections and features
import cartopy.crs as ccrs
import cartopy.feature as cfeature
# from cartopy.io.shapereader import Reader as ShapeReader # Not strictly needed if using cfeature

print("--- 1. COMMON DATA LOADING AND PREPARATION ---")

# --- Configuration ---
BASE_DIR = '../metadata/' # Adjust if your script is in a different location
ATTRIBUTES_PATH = os.path.join(BASE_DIR, 'attributes_backup', 'attributes.csv')
CSV_FILTERED_DIR = os.path.join(BASE_DIR, 'csv_filtered')
SHAPEFILE_PATH = os.path.join(BASE_DIR, 'shapefile', 'ADP02_Basin_Select.shp')

COLS_TO_LOAD_ATTR = [
    'gauge_id', 'area', 'pre_mm_syr', 'tmp_dc_syr', 'dis_m3_pyr',
    'for_pc_sse', 'crp_pc_sse', 'urb_pc_sse', 'wet_pc_sg1', 'lka_pc_sse',
    # 'glc_pc_s01'
]
SHP_GAUGE_ID_COL = 'StationNum'

# --- Load and Prepare Data ---
basin_csv_files = glob.glob(os.path.join(CSV_FILTERED_DIR, '*.csv'))
studied_basin_ids = [os.path.splitext(os.path.basename(f))[0] for f in basin_csv_files]
print(f"Found {len(studied_basin_ids)} studied basin IDs from CSV files.")

try:
    attributes_df = pd.read_csv(ATTRIBUTES_PATH, usecols=COLS_TO_LOAD_ATTR)
    attributes_df['gauge_id'] = attributes_df['gauge_id'].astype(str)
except ValueError as e:
    print(f"Error loading attributes CSV. Check COLS_TO_LOAD_ATTR. Details: {e}")
    # Attempt to load all columns for inspection if error occurs
    temp_df_inspect = pd.read_csv(ATTRIBUTES_PATH, nrows=5)
    print(f"Available columns in attributes.csv: {temp_df_inspect.columns.tolist()}")
    exit()


attributes_study_df = attributes_df[attributes_df['gauge_id'].isin(studied_basin_ids)].copy()
print(f"Attributes data filtered to {len(attributes_study_df)} studied basins.")

if attributes_study_df.empty:
    print("No matching basins found between attributes.csv and filtered CSV files. Exiting.")
    exit()

gdf_all = gpd.read_file(SHAPEFILE_PATH)

--- 1. COMMON DATA LOADING AND PREPARATION ---
Found 976 studied basin IDs from CSV files.
Attributes data filtered to 976 studied basins.


In [8]:
gdf_all

,StationNum,NameNom,Status,Etat,Area_km2,Aire_km2,Remark,Remarque,Version,Date_rev,Shape_Leng,Shape_Area,geometry
0,02AA001,PIGEON RIVER AT MIDDLE FALLS,discontinued,fermée,1573.7200,1573.7200,None,None,June 2024 / juin 2024,2024-06-01,366480.0,1.573723e+09,"POLYGON ((447750 883800, 447780 883800, 447780..."
1,02AA002,PINE RIVER NEAR CROOKS,discontinued,fermée,392.4500,392.4500,None,None,June 2024 / juin 2024,2024-06-01,176700.0,3.924504e+08,"POLYGON ((469740 906510, 469800 906510, 469800..."
2,02AB001,KAMINISTIQUIA RIVER NEAR DONA,discontinued,fermée,3565.1500,3565.1500,None,None,June 2024 / juin 2024,2024-06-01,766140.0,3.565148e+09,"POLYGON ((465420 976290, 465420 976320, 465450..."
3,02AB002,SHEBANDOWAN RIVER NEAR KAMINISTIQUIA,discontinued,fermée,2902.9100,2902.9100,None,None,June 2024 / juin 2024,2024-06-01,546900.0,2.902905e+09,"POLYGON ((465420 976290, 465390 976290, 465390..."
4,02AB003,KAMINISTIQUIA RIVER AT MOKOMON,discontinued,fermée,6644.5700,6644.5700,None,None,June 2024 / juin 2024,2024-06-01,1141320.0,6.644574e+09,"POLYGON ((422190 1027230, 422190 1027200, 4222..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
644,02HH006,STURGEON LAKE AT FENELON FALLS,active,en service,3253.8900,3253.8900,None,None,June 2024 / juin 2024,2024-06-01,663960.0,3.253890e+09,"POLYGON ((1363470 652410, 1363380 652410, 1363..."
645,02HJ011,OUSE RIVER AT NORWOOD,active,en service,90.9171,90.9171,None,None,June 2024 / juin 2024,2024-06-01,102120.0,9.091710e+07,"POLYGON ((1440660 670320, 1440630 670320, 1440..."
646,02HJ012,INDIAN RIVER BELOW HOPE MILL DAM,active,en service,150.9410,150.9410,None,None,June 2024 / juin 2024,2024-06-01,136800.0,1.509408e+08,"POLYGON ((1429110 655290, 1429080 655290, 1429..."
647,02HK018,TRENT RIVER AT UPPER GLEN ROSS,active,en service,12005.4000,12005.4000,None,None,June 2024 / juin 2024,2024-06-01,1136220.0,1.200538e+10,"POLYGON ((1363380 650670, 1363380 650700, 1363..."


In [13]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D # For custom legends
import numpy as np
import os
import glob

# For map projections and features
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print("--- 1. COMMON DATA LOADING AND PREPARATION (US & Canada) - CORRECTED ID COLS ---")

# --- Configuration ---
BASE_DIR = '../metadata/'
ATTRIBUTES_PATH = os.path.join(BASE_DIR, 'attributes_backup', 'attributes.csv')
CSV_FILTERED_DIR = os.path.join(BASE_DIR, 'csv_filtered')

# CORRECTED Shapefile ID Column Names
US_SHAPEFILE_PATH = os.path.join(BASE_DIR, 'shapefile', 'GL_GAGE2_all.shp')
US_SHP_GAUGE_ID_COL = 'GAGE_ID' # As per your clarification for US stations

CA_SHAPEFILE_PATH = os.path.join(BASE_DIR, 'shapefile', 'ADP02_Basin_Select.shp')
CA_SHP_GAUGE_ID_COL = 'StationNum' # VERIFY this is correct for Canadian. If it's 'StationName', change it.

COLS_TO_LOAD_ATTR = [
    'gauge_id', 'area', 'pre_mm_syr', 'tmp_dc_syr', 'dis_m3_pyr',
    'for_pc_sse', 'crp_pc_sse', 'urb_pc_sse', 'wet_pc_sg1', 'lka_pc_sse',
]

# --- Load and Prepare Data ---
basin_csv_files = glob.glob(os.path.join(CSV_FILTERED_DIR, '*.csv'))
studied_basin_ids = [os.path.splitext(os.path.basename(f))[0] for f in basin_csv_files]
print(f"Found {len(studied_basin_ids)} studied basin IDs from CSV files.")

try:
    attributes_df = pd.read_csv(ATTRIBUTES_PATH, usecols=COLS_TO_LOAD_ATTR)
    attributes_df['gauge_id'] = attributes_df['gauge_id'].astype(str).str.strip()
    print(f"Attributes CSV loaded: {len(attributes_df)} rows.")
except ValueError as e:
    print(f"Error loading attributes CSV: {e}")
    exit()

attributes_study_df = attributes_df[attributes_df['gauge_id'].isin(studied_basin_ids)].copy()
print(f"Attributes data filtered to {len(attributes_study_df)} studied basins.")
if attributes_study_df.empty:
    print("No matching basins found between attributes.csv and filtered CSV files. Exiting.")
    exit()

# --- Load US Shapefile ---
gdf_us = None
try:
    gdf_us_raw = gpd.read_file(US_SHAPEFILE_PATH)
    print(f"US Shapefile '{US_SHAPEFILE_PATH}' loaded with {len(gdf_us_raw)} features. CRS: {gdf_us_raw.crs}")
    if US_SHP_GAUGE_ID_COL not in gdf_us_raw.columns:
        print(f"ERROR: Column '{US_SHP_GAUGE_ID_COL}' not found in US shapefile. Available: {gdf_us_raw.columns.tolist()}")
    else:
        gdf_us = gdf_us_raw[[US_SHP_GAUGE_ID_COL, 'geometry']].copy()
        gdf_us = gdf_us.rename(columns={US_SHP_GAUGE_ID_COL: 'gauge_id_shp'}) # Standardize to 'gauge_id_shp'
        gdf_us['gauge_id_shp'] = gdf_us['gauge_id_shp'].astype(str).str.strip()
        print(f"US GDF processed. Length: {len(gdf_us)}. Target ID col: 'gauge_id_shp'")
except Exception as e:
    print(f"Error loading or processing US Shapefile: {e}")

# --- Load Canadian Shapefile ---
gdf_ca = None
try:
    gdf_ca_raw = gpd.read_file(CA_SHAPEFILE_PATH)
    print(f"Canadian Shapefile '{CA_SHAPEFILE_PATH}' loaded with {len(gdf_ca_raw)} features. CRS: {gdf_ca_raw.crs}")
    if CA_SHP_GAUGE_ID_COL not in gdf_ca_raw.columns:
        print(f"ERROR: Column '{CA_SHP_GAUGE_ID_COL}' not found in Canadian shapefile. Available: {gdf_ca_raw.columns.tolist()}")
    else:
        gdf_ca = gdf_ca_raw[[CA_SHP_GAUGE_ID_COL, 'geometry']].copy()
        gdf_ca = gdf_ca.rename(columns={CA_SHP_GAUGE_ID_COL: 'gauge_id_shp'}) # Standardize to 'gauge_id_shp'
        gdf_ca['gauge_id_shp'] = gdf_ca['gauge_id_shp'].astype(str).str.strip()
        print(f"Canadian GDF processed. Length: {len(gdf_ca)}. Target ID col: 'gauge_id_shp'")
except Exception as e:
    print(f"Error loading or processing Canadian Shapefile: {e}")

# --- Concatenate US and Canadian GeoDataFrames ---
# (Rest of the common data preparation block remains the same as the previous "DEBUGGING" version)
gdf_all_countries_list = []
common_crs_for_concat = None

if gdf_us is not None and not gdf_us.empty:
    common_crs_for_concat = gdf_us.crs
    gdf_all_countries_list.append(gdf_us)
    print(f"US GDF added to list for concatenation. CRS: {gdf_us.crs}")

if gdf_ca is not None and not gdf_ca.empty:
    if common_crs_for_concat is None:
        common_crs_for_concat = gdf_ca.crs
    if gdf_ca.crs != common_crs_for_concat:
        print(f"Reprojecting Canadian GDF from {gdf_ca.crs} to {common_crs_for_concat} for concatenation.")
        try:
            gdf_ca = gdf_ca.to_crs(common_crs_for_concat)
            print(f"Canadian GDF reprojected. New CRS: {gdf_ca.crs}")
        except Exception as e:
            print(f"Error reprojecting Canadian GDF: {e}. Skipping Canadian data for concatenation.")
            gdf_ca = None
    if gdf_ca is not None and not gdf_ca.empty:
         gdf_all_countries_list.append(gdf_ca)
         print("Canadian GDF added to list for concatenation.")

if not gdf_all_countries_list:
    print("ERROR: No shapefile data available for concatenation. Exiting.")
    exit()

gdf_all_countries = gpd.GeoDataFrame(pd.concat(gdf_all_countries_list, ignore_index=True), crs=common_crs_for_concat)
print(f"Combined shapefiles. Total {len(gdf_all_countries)} features. Initial Combined CRS: {gdf_all_countries.crs}")
print(f"Number of unique gauge_id_shp in combined GDF: {gdf_all_countries['gauge_id_shp'].nunique()}")

print(f"Merging gdf_all_countries (gauge_id_shp) with attributes_study_df (gauge_id)")
gdf_study = gdf_all_countries.merge(attributes_study_df, left_on='gauge_id_shp', right_on='gauge_id', how='inner')
print(f"Merged combined shapefile with attributes, resulting in {len(gdf_study)} features for study.")

# Debugging the merge (uncomment if needed):
# common_ids = set(gdf_all_countries['gauge_id_shp'].unique()) & set(attributes_study_df['gauge_id'].unique())
# print(f"Number of common IDs for merge: {len(common_ids)}")
# if len(common_ids) < 10 and len(common_ids) > 0: print(f"Sample common IDs: {list(common_ids)[:min(10, len(common_ids))]}")
# else:
#     print("Sample IDs from gdf_all_countries['gauge_id_shp']:", list(gdf_all_countries['gauge_id_shp'].unique()[:10]))
#     print("Sample IDs from attributes_study_df['gauge_id']:", list(attributes_study_df['gauge_id'].unique()[:10]))


if gdf_study.empty:
    print("No features after merging. This means 'gauge_id_shp' from the combined shapefiles did not match 'gauge_id' in your attributes for the studied basins.")
    exit()

TARGET_CRS = 'EPSG:4326'
if gdf_study.crs != TARGET_CRS:
    print(f"Reprojecting final GDF from {gdf_study.crs} to {TARGET_CRS}")
    gdf_study = gdf_study.to_crs(TARGET_CRS)
else:
    print(f"Final GDF already in target CRS: {TARGET_CRS}")

study_domain_boundary = gdf_study.unary_union
domain_boundary_gdf = gpd.GeoDataFrame(geometry=[study_domain_boundary], crs=gdf_study.crs)

def get_dominant_land_cover(row):
    lc_categories = {
        'Forest': row.get('for_pc_sse', 0), 'Agriculture': row.get('crp_pc_sse', 0),
        'Urban': row.get('urb_pc_sse', 0), 'Wetland': row.get('wet_pc_sg1', 0),
        'Open Water': row.get('lka_pc_sse', 0),
    }
    valid_lc = {k: v for k, v in lc_categories.items() if pd.notna(v)}
    if not valid_lc: return 'Unknown'
    dominant = max(valid_lc, key=valid_lc.get)
    return dominant

required_lc_cols = ['for_pc_sse', 'crp_pc_sse', 'urb_pc_sse', 'wet_pc_sg1', 'lka_pc_sse']
if all(col in gdf_study.columns for col in required_lc_cols):
    gdf_study['dominant_lc'] = gdf_study.apply(get_dominant_land_cover, axis=1)
    lc_colors = {
        'Forest': 'darkgreen', 'Agriculture': 'gold', 'Urban': 'gray',
        'Wetland': 'mediumblue', 'Open Water': 'lightblue', 'Unknown': 'white'
    }
    print("Dominant land cover calculated. Counts:\n", gdf_study['dominant_lc'].value_counts())
else:
    missing_cols_lc = [col for col in required_lc_cols if col not in gdf_study.columns]
    print(f"Skipping dominant land cover (missing columns: {missing_cols_lc}).")
    gdf_study['dominant_lc'] = 'Not Calculated'
    lc_colors = {'Not Calculated': 'white'}

great_lakes_labels = {
    'Lake Superior': (-88.0, 47.5), 'Lake Michigan': (-87.0, 44.0),
    'Lake Huron': (-82.5, 44.5), 'Lake Erie': (-81.0, 42.2),
    'Lake Ontario': (-77.5, 43.7),
}

map_extent = [-94, -73, 39.5, 50]
print(f"Using MANUALLY set map extent: {map_extent}")

def setup_great_lakes_map(ax_map, extent):
    ax_map.set_extent(extent, crs=ccrs.PlateCarree())
    ax_map.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#E0E0E0', edgecolor='gray', zorder=0)
    ax_map.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='aliceblue', zorder=0)
    ax_map.add_feature(cfeature.LAKES.with_scale('10m'), facecolor='aliceblue', edgecolor='darkgray', zorder=1)
    ax_map.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':', edgecolor='black', linewidth=0.7, zorder=2)
    ax_map.add_feature(cfeature.STATES.with_scale('50m'), linestyle=':', edgecolor='dimgray', linewidth=0.5, zorder=2)
    for name, (lon, lat) in great_lakes_labels.items():
        ax_map.text(lon, lat, name, transform=ccrs.Geodetic(),
                    ha='center', va='center', fontsize=7, fontweight='bold',
                    bbox=dict(facecolor='white', alpha=0.5, pad=0.1, edgecolor='none'), zorder=5)
    return ax_map

print("--- COMMON DATA PREPARATION COMPLETE (US & Canada) - CORRECTED ID COLS ---")

--- 1. COMMON DATA LOADING AND PREPARATION (US & Canada) - CORRECTED ID COLS ---
Found 976 studied basin IDs from CSV files.
Attributes CSV loaded: 1088 rows.
Attributes data filtered to 976 studied basins.
US Shapefile '../metadata/shapefile\GL_GAGE2_all.shp' loaded with 439 features. CRS: PROJCS["NAD_1983_Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
US GDF processed. Length: 439. Target ID col: 'gauge_id_shp'
Canadian Shapefile '../metadata/shapefile\ADP02_Basin_Select.shp' loaded with 649 features.

C:\Users\ybrot\AppData\Local\Temp\ipykernel_420636\2449951724.py:139: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  study_domain_boundary = gdf_study.unary_union


Dominant land cover calculated. Counts:
 dominant_lc
Agriculture    361
Forest         339
Open Water     208
Urban           56
Wetland         12
Name: count, dtype: int64
Using MANUALLY set map extent: [-94, -73, 39.5, 50]
--- COMMON DATA PREPARATION COMPLETE (US & Canada) - CORRECTED ID COLS ---


In [14]:
print("\n--- GENERATING FIGURE 1: Study Domain and Land Cover (US & Canada) ---")

# Ensure the common block has been run and gdf_study, domain_boundary_gdf are available
if 'gdf_study' not in globals() or 'domain_boundary_gdf' not in globals():
    print("ERROR: Common data preparation block not run. Please run it first.")
else:
    fig1, (ax1a, ax1b) = plt.subplots(1, 2, figsize=(16, 8),
                                   # Use LambertConformal as the projection for display
                                   subplot_kw={'projection': ccrs.LambertConformal(central_longitude=-85, central_latitude=45)})

    # Panel A: Geographic Location of Study Sub-basins
    ax1a = setup_great_lakes_map(ax1a, map_extent) # map_extent is already in lat/lon
    ax1a.set_title('A) Geographic Location of Studied Sub-basins', fontsize=11)
    # Data is in EPSG:4326 (PlateCarree), so specify transform for plotting
    gdf_study.plot(ax=ax1a, transform=ccrs.PlateCarree(),
                   facecolor='skyblue', edgecolor='darkblue', linewidth=0.3, alpha=0.6, zorder=3)
    domain_boundary_gdf.plot(ax=ax1a, transform=ccrs.PlateCarree(),
                             facecolor='none', edgecolor='red', linewidth=1.2, linestyle='--', zorder=4)
    handles_a = [
        Line2D([0], [0], marker='s', color='w', label='Studied Sub-basins Extent', markerfacecolor='skyblue', markeredgecolor='darkblue', markersize=8),
        Line2D([0], [0], color='red', lw=1.2, linestyle='--', label='Overall Study Domain Boundary')
    ]
    ax1a.legend(handles=handles_a, loc='lower left', fontsize=8)
    ax1a.text(0.01, 0.01, '(Basemap: Natural Earth)', transform=ax1a.transAxes, fontsize=6, ha='left', va='bottom', color='dimgray')


    # Panel B: Generalized Land Cover of Studied Sub-basins
    ax1b = setup_great_lakes_map(ax1b, map_extent)
    ax1b.set_title('B) Dominant Land Cover of Studied Sub-basins', fontsize=11)
    legend_elements_lc = []
    if gdf_study['dominant_lc'].nunique() > 1 and gdf_study['dominant_lc'].iloc[0] != 'Not Calculated':
        for lc_type in gdf_study['dominant_lc'].unique(): # Iterate unique values to maintain order
            if lc_type == 'Not Calculated' or lc_type == 'Unknown': continue # Skip these for plotting if desired
            color = lc_colors.get(lc_type, 'purple')
            gdf_subset = gdf_study[gdf_study['dominant_lc'] == lc_type]
            if not gdf_subset.empty:
                gdf_subset.plot(ax=ax1b, transform=ccrs.PlateCarree(),
                                facecolor=color, edgecolor='black', linewidth=0.1, label=lc_type, zorder=3)
                legend_elements_lc.append(Line2D([0], [0], marker='s', color='w', label=lc_type,
                                              markerfacecolor=color, markersize=8))
        if legend_elements_lc: # Only add legend if there are items
            ax1b.legend(handles=legend_elements_lc, title="Dominant Land Cover", loc='lower left', fontsize=8, title_fontsize=9)
    else:
        # Fallback if land cover isn't calculated or is uniform
        gdf_study.plot(ax=ax1b, transform=ccrs.PlateCarree(), facecolor='lightgrey', edgecolor='black', linewidth=0.1, zorder=3)
        ax1b.text(0.5, 0.5, "Dominant Land Cover Not Plotted\n(Data missing or uniform)",
                  transform=ax1b.transAxes, ha='center', va='center', color='red', fontsize=9)
    ax1b.text(0.01, 0.01, '(Basemap: Natural Earth)', transform=ax1b.transAxes, fontsize=6, ha='left', va='bottom', color='dimgray')


    plt.tight_layout(pad=1.5)
    plt.savefig('Figure1_Study_Domain_Land_Cover_Combined.png', dpi=300, bbox_inches='tight')
    print("Figure 1 saved as Figure1_Study_Domain_Land_Cover_Combined.png")
    plt.close(fig1)


--- GENERATING FIGURE 1: Study Domain and Land Cover (US & Canada) ---


C:\Users\ybrot\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\cartopy\io\__init__.py:241: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_lakes.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


Figure 1 saved as Figure1_Study_Domain_Land_Cover_Combined.png


In [17]:
import pandas as pd # Ensure pandas is imported if not already from common block
import numpy as np # Ensure numpy is imported if not already from common block
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
# Assuming setup_great_lakes_map, map_extent, and gdf_study are available from the common block

print("\n--- GENERATING FIGURE 2: Climatic Gradients (with F to C conversion) ---")

if 'gdf_study' not in globals() or gdf_study.empty:
    print("ERROR: Common data preparation block not run or gdf_study is empty. Please run it first.")
else:
    fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(16, 8),
                                   subplot_kw={'projection': ccrs.LambertConformal(central_longitude=-85, central_latitude=45)})

    # Create a copy for modifications to avoid SettingWithCopyWarning if gdf_study is a slice
    gdf_plot_fig2 = gdf_study.copy()

    # Panel A: Mean Annual Precipitation
    ax2a = setup_great_lakes_map(ax2a, map_extent) # setup_great_lakes_map should be defined in your common block
    ax2a.set_title('A) Mean Annual Precipitation (mm/year)', fontsize=11)
    if 'pre_mm_syr' in gdf_plot_fig2.columns:
        gdf_plot_fig2.plot(ax=ax2a, transform=ccrs.PlateCarree(), column='pre_mm_syr',
                           cmap='Blues', legend=True,
                           legend_kwds={'label': "Precipitation (mm/year)", 'orientation': "horizontal", 'shrink': 0.5, 'pad': 0.1, 'aspect': 30},
                           edgecolor='darkgray', linewidth=0.1, missing_kwds={'color': 'lightgrey', "hatch": "///", "label": "Missing values"})
    else:
        ax2a.text(0.5, 0.5, "Precipitation Data ('pre_mm_syr') Not Found", transform=ax2a.transAxes, ha='center', va='center', color='red', fontsize=9)
    ax2a.text(0.01, 0.01, '(Basemap: Natural Earth)', transform=ax2a.transAxes, fontsize=6, ha='left', va='bottom', color='dimgray')

    # Panel B: Mean Annual Air Temperature
    ax2b = setup_great_lakes_map(ax2b, map_extent)
    ax2b.set_title('B) Mean Annual Air Temperature (°C)', fontsize=11)
    if 'tmp_dc_syr' in gdf_plot_fig2.columns:
        # Convert Fahrenheit to Celsius
        # Ensure the column is numeric before conversion
        if pd.api.types.is_numeric_dtype(gdf_plot_fig2['tmp_dc_syr']):
            gdf_plot_fig2['tmp_dc_syr_C'] = (gdf_plot_fig2['tmp_dc_syr'] - 32) * 5/9
            print("Temperature column 'tmp_dc_syr' converted from F to C as 'tmp_dc_syr_C'.")

            gdf_plot_fig2.plot(ax=ax2b, transform=ccrs.PlateCarree(), column='tmp_dc_syr_C', # Plot the new Celsius column
                               cmap='coolwarm', legend=True,
                               legend_kwds={'label': "Temperature (°C)", 'orientation': "horizontal", 'shrink': 0.5, 'pad': 0.1, 'aspect': 30},
                               edgecolor='darkgray', linewidth=0.1, missing_kwds={'color': 'lightgrey', "hatch": "///", "label": "Missing values"})
        else:
            print("ERROR: Temperature column 'tmp_dc_syr' is not numeric. Cannot convert F to C.")
            ax2b.text(0.5, 0.5, "Temperature Data ('tmp_dc_syr')\nNot Numeric for Conversion", transform=ax2b.transAxes, ha='center', va='center', color='red', fontsize=9)
    else:
        ax2b.text(0.5, 0.5, "Temperature Data ('tmp_dc_syr') Not Found", transform=ax2b.transAxes, ha='center', va='center', color='red', fontsize=9)
    ax2b.text(0.01, 0.01, '(Basemap: Natural Earth)', transform=ax2b.transAxes, fontsize=6, ha='left', va='bottom', color='dimgray')

    plt.tight_layout(pad=1.5)
    plt.savefig('Figure2_Climatic_Gradients_Celsius.png', dpi=300, bbox_inches='tight')
    print("Figure 2 saved as Figure2_Climatic_Gradients_Celsius.png")
    plt.close(fig2)


--- GENERATING FIGURE 2: Climatic Gradients (with F to C conversion) ---
Temperature column 'tmp_dc_syr' converted from F to C as 'tmp_dc_syr_C'.
Figure 2 saved as Figure2_Climatic_Gradients_Celsius.png


In [16]:
print("\n--- GENERATING FIGURE 3: Distribution of Key Basin Characteristics ---")

if 'gdf_study' not in globals():
    print("ERROR: Common data preparation block not run. Please run it first.")
else:
    fig3, (ax3a, ax3b) = plt.subplots(1, 2, figsize=(10, 4)) # Smaller figure

    # Panel A: Drainage Areas
    if 'area' in gdf_study.columns:
        areas_km2 = gdf_study['area'].dropna()
        if not areas_km2.empty and areas_km2.min() > 0 :
            ax3a.hist(np.log10(areas_km2), bins=25, color='skyblue', edgecolor='black')
            ax3a.set_xlabel('Log₁₀(Drainage Area [km²])')
        elif not areas_km2.empty:
            ax3a.hist(areas_km2, bins=25, color='skyblue', edgecolor='black')
            ax3a.set_xlabel('Drainage Area (km²)')
        else:
            ax3a.text(0.5, 0.5, "Area Data Missing", transform=ax3a.transAxes, ha='center', va='center', color='red')
    else:
        ax3a.text(0.5, 0.5, "Area Column Missing", transform=ax3a.transAxes, ha='center', va='center', color='red')
    ax3a.set_ylabel('Number of Basins')
    ax3a.set_title('A) Distribution of Drainage Areas', fontsize=10)
    ax3a.grid(axis='y', linestyle='--', alpha=0.7)
    ax3a.tick_params(axis='both', which='major', labelsize=8)


    # Panel B: Mean Annual Discharge
    if 'dis_m3_pyr' in gdf_study.columns:
        seconds_in_year = 365.25 * 24 * 60 * 60
        discharge_m3_s = (gdf_study['dis_m3_pyr'] / seconds_in_year).dropna()
        if not discharge_m3_s.empty and discharge_m3_s.min() > 0:
            ax3b.hist(np.log10(discharge_m3_s), bins=25, color='lightcoral', edgecolor='black')
            ax3b.set_xlabel('Log₁₀(Mean Annual Discharge [m³/s])')
        elif not discharge_m3_s.empty:
            ax3b.hist(discharge_m3_s, bins=25, color='lightcoral', edgecolor='black')
            ax3b.set_xlabel('Mean Annual Discharge (m³/s)')
        else:
            ax3b.text(0.5, 0.5, "Discharge Data Missing", transform=ax3b.transAxes, ha='center', va='center', color='red')
    else:
        ax3b.text(0.5, 0.5, "Discharge Column Missing", transform=ax3b.transAxes, ha='center', va='center', color='red')

    ax3b.set_ylabel('Number of Basins')
    ax3b.set_title('B) Distribution of Mean Annual Discharge', fontsize=10)
    ax3b.grid(axis='y', linestyle='--', alpha=0.7)
    ax3b.tick_params(axis='both', which='major', labelsize=8)


    plt.tight_layout(pad=1.0)
    plt.savefig('Figure3_Basin_Characteristics_Distribution.png', dpi=300, bbox_inches='tight')
    print("Figure 3 saved as Figure3_Basin_Characteristics_Distribution.png")
    plt.close(fig3)

print("\n--- ALL FIGURE GENERATION ATTEMPTED ---")


--- GENERATING FIGURE 3: Distribution of Key Basin Characteristics ---
Figure 3 saved as Figure3_Basin_Characteristics_Distribution.png

--- ALL FIGURE GENERATION ATTEMPTED ---
